# ER Surge & Bed-Risk Advisor
### Google Cloud Gen AI Academy Hackathon — NVIDIA Accelerated Data Intelligence Track

**User:** ED charge nurse / shift supervisor
**Decision supported:** divert patients / call in extra staff / open overflow beds
**Pipeline:** Cloud Storage/BigQuery → feature engineering (wait-time stats, rolling averages) → risk scoring → dashboard-ready output
**Acceleration proof:** same pipeline run in pandas vs. `cudf.pandas` (GPU), benchmarked on a large dataset

This notebook is **Step 1** of the build plan: get the pandas pipeline working end-to-end on a small dataset in a GPU-enabled Colab runtime. It's built against `Hospital_ER_Data.csv` (9,216 real records, Apr 2023-Oct 2024) and written so Step 2 (porting to `cudf.pandas`) is a near-zero-code-change flip -- see the toggle cell below.

**Before running:** In Colab, go to `Runtime -> Change runtime type -> T4 GPU` (or better) and save. Then upload `Hospital_ER_Data.csv` to `/content/`.

**A note on this dataset's limitations (say this in the deck, don't hide it):** it has no triage/acuity column and no bed-capacity column. The risk score below is built entirely from proxies (admission flag, case-management flag, relative arrival volume) rather than those fields. That's a legitimate, explainable v1 -- just be upfront that a production version would plug in a real census/staffing feed via Cloud Storage or BigQuery where `beds_occupied`/`triage_level` would normally live.


## 0. Environment check
Confirm we actually have a GPU attached -- this matters later for the cuDF benchmark, and cudf.pandas will silently no-op without one.

In [ ]:
!nvidia-smi


In [ ]:
# One-time install if you haven't already got RAPIDS in this Colab runtime.
# Safe to run now even though we're using plain pandas in this notebook --
# it just means Step 2 (cuDF port) won't need a fresh environment.
# Uncomment on first run:

# !pip install --extra-index-url=https://pypi.nvidia.com cudf-cu12 -q


## 1. Config: pandas vs. cudf.pandas toggle

This is the switch we'll flip for the Step 2 benchmark. `cudf.pandas` is a **zero-code-change** GPU accelerator for pandas -- loading it as an IPython extension before `import pandas as pd` transparently routes supported operations to the GPU. Everything below is written against the plain `pd` alias so flipping `USE_CUDF = True` later doesn't require touching the pipeline code, only this cell.


In [ ]:
USE_CUDF = False  # flip to True in Step 2, after `pip install cudf-cu12`

if USE_CUDF:
    %load_ext cudf.pandas

import pandas as pd
import numpy as np

print(f"USE_CUDF = {USE_CUDF}")
print(f"pandas module: {pd.__name__}")


## 2. Data ingestion

Loads the real Kaggle-style file `Hospital_ER_Data.csv`. Raw columns:

`Patient Id, Patient Admission Date, Patient First Inital, Patient Last Name, Patient Gender, Patient Age, Patient Race, Department Referral, Patient Admission Flag, Patient Satisfaction Score, Patient Waittime, Patients CM`

Two data quality issues handled explicitly below:
- `Department Referral` is **58% missing** -- those patients were treated in the general ED rather than referred to a specialty department, so missing is filled as `"General ER"`, not dropped.
- `Patient Satisfaction Score` is **73% missing** -- not used in the risk score at all, since it's too sparse to trust as a rolling signal.

If no file is found, a small synthetic dataset with the same schema is generated so the notebook still runs end-to-end.


In [ ]:
import os
from datetime import datetime, timedelta

CSV_PATH = "/content/Hospital_ER_Data.csv"

def load_er_data(path=CSV_PATH, n_synth=6000, seed=42):
    if os.path.exists(path):
        print(f"Loading real dataset from {path}")
        raw = pd.read_csv(path)
    else:
        print(f"No file found at {path} -- generating {n_synth} synthetic records matching the same schema.")
        rng = np.random.default_rng(seed)
        depts = ["General Practice", "Orthopedics", "Physiotherapy", "Cardiology",
                  "Neurology", "Gastroenterology", "Renal", np.nan]
        dept_weights = [0.20, 0.11, 0.03, 0.03, 0.02, 0.02, 0.01, 0.58]
        start = datetime(2023, 4, 1)
        offsets = rng.integers(0, 570 * 24 * 60, size=n_synth)
        raw = pd.DataFrame({
            "Patient Id": [f"{rng.integers(100,999)}-{rng.integers(10,99)}-{rng.integers(1000,9999)}" for _ in range(n_synth)],
            "Patient Admission Date": [(start + timedelta(minutes=int(m))).strftime("%d-%m-%Y %H:%M") for m in offsets],
            "Patient Gender": rng.choice(["M", "F"], size=n_synth),
            "Patient Age": rng.integers(0, 95, size=n_synth),
            "Department Referral": rng.choice(depts, size=n_synth, p=dept_weights),
            "Patient Admission Flag": rng.choice([True, False], size=n_synth),
            "Patient Satisfaction Score": rng.choice(list(range(1,11)) + [np.nan]*3, size=n_synth),
            "Patient Waittime": rng.integers(10, 61, size=n_synth),
            "Patients CM": rng.choice([0, 1], size=n_synth, p=[0.85, 0.15]),
        })

    df = pd.DataFrame()
    df["patient_id"] = raw["Patient Id"]
    df["arrival_time"] = pd.to_datetime(raw["Patient Admission Date"], format="%d-%m-%Y %H:%M")
    df["department"] = raw["Department Referral"].fillna("General ER")
    df["age"] = raw["Patient Age"]
    df["wait_time_min"] = raw["Patient Waittime"].astype(float)
    df["admitted"] = raw["Patient Admission Flag"].astype(int)
    df["case_mgmt_flag"] = raw["Patients CM"].astype(int)
    return df

df = load_er_data()
print(df.shape)
df.head()


## 3. Feature engineering

This is the part that gets ported to `cudf.pandas` unchanged in Step 2. Built entirely from the columns that actually exist:
- Time bucketing (hour, day-of-week) so surge patterns show up
- Rolling wait-time average per department (the leading indicator a charge nurse watches)
- **Relative arrival load**: arrivals this hour vs. that department's own historical average hourly volume -- this is our load/capacity-pressure proxy, since there's no real bed-count column
- Rolling admission rate per department (higher admit rate = sicker recent patients = acuity proxy)
- Rolling case-management-flag rate (complexity proxy)


In [ ]:
df = df.sort_values(["department", "arrival_time"]).reset_index(drop=True)

df["hour"] = df["arrival_time"].dt.floor("h")
df["day_of_week"] = df["arrival_time"].dt.day_name()
df["hour_of_day"] = df["arrival_time"].dt.hour

df["rolling_wait_avg_10"] = (
    df.groupby("department")["wait_time_min"]
      .transform(lambda s: s.rolling(window=10, min_periods=1).mean())
)

arrivals_per_hour = (
    df.groupby(["department", "hour"]).size().rename("arrivals_this_hour").reset_index()
)
df = df.merge(arrivals_per_hour, on=["department", "hour"], how="left")

dept_baseline = (
    arrivals_per_hour.groupby("department")["arrivals_this_hour"]
      .mean().rename("dept_baseline_arrivals").reset_index()
)
df = df.merge(dept_baseline, on="department", how="left")
df["relative_load"] = df["arrivals_this_hour"] / df["dept_baseline_arrivals"]

df["rolling_admit_rate_10"] = (
    df.groupby("department")["admitted"]
      .transform(lambda s: s.rolling(window=10, min_periods=1).mean())
)
df["rolling_cm_rate_10"] = (
    df.groupby("department")["case_mgmt_flag"]
      .transform(lambda s: s.rolling(window=10, min_periods=1).mean())
)

df[["department", "arrival_time", "wait_time_min", "rolling_wait_avg_10",
    "relative_load", "rolling_admit_rate_10", "rolling_cm_rate_10"]].head(10)


## 4. Risk scoring

A simple, explainable weighted score -- charge nurses need to audit this, not trust a black box. Each component is normalized 0-1 across the current data, then combined. Weights are a v1 judgment call; wait time and relative load matter most for an immediate staffing/diversion decision.


In [ ]:
def normalize(s):
    lo, hi = s.min(), s.max()
    if hi - lo == 0:
        return s * 0
    return (s - lo) / (hi - lo)

df["risk_wait"] = normalize(df["rolling_wait_avg_10"])
df["risk_load"] = normalize(df["relative_load"])
df["risk_acuity"] = normalize(df["rolling_admit_rate_10"])
df["risk_complexity"] = normalize(df["rolling_cm_rate_10"])

WEIGHTS = {"risk_wait": 0.30, "risk_load": 0.30, "risk_acuity": 0.25, "risk_complexity": 0.15}
df["bed_risk_score"] = sum(df[col] * w for col, w in WEIGHTS.items())

# Data-driven thresholds rather than fixed cutoffs: the top 10% of observed
# risk scores are flagged High, the next 25% Medium, the rest Low. This adapts
# to each hospital's own historical distribution instead of assuming a fixed
# scale -- a fixed 0.70 cutoff looked reasonable on paper but never triggered
# on this dataset's actual score range (max ~0.64).
RISK_THRESHOLD_HIGH = df["bed_risk_score"].quantile(0.90)
RISK_THRESHOLD_MED = df["bed_risk_score"].quantile(0.65)

df["risk_tier"] = pd.cut(
    df["bed_risk_score"],
    bins=[-0.01, RISK_THRESHOLD_MED, RISK_THRESHOLD_HIGH, 1.01],
    labels=["Low", "Medium", "High"]
)

print(f"Thresholds -- Medium >= {RISK_THRESHOLD_MED:.3f}, High >= {RISK_THRESHOLD_HIGH:.3f}")
print(df["risk_tier"].value_counts())
df[["department", "arrival_time", "bed_risk_score", "risk_tier"]].sort_values("bed_risk_score", ascending=False).head(10)


**Note on thresholds**: these are quantile-based (top 10%/next 25%/rest), not fixed values -- so tiers will always show a meaningful spread regardless of the dataset's raw score range. Recompute them whenever you swap in a new or larger dataset (e.g. after moving to the CMS Medicare data in Step 2), since the quantiles shift with the data.


## 5. Dashboard-ready output

Aggregate to (department x hour) -- the grain a charge nurse actually scans: *"is General ER about to surge in the next hour?"* This is what the dashboard/UI layer (Step 3) renders, and the artifact exported for the demo.


In [ ]:
dashboard_view = (
    df.groupby(["department", "hour"])
      .agg(
          avg_wait_min=("wait_time_min", "mean"),
          patient_count=("patient_id", "count"),
          avg_admit_rate=("admitted", "mean"),
          avg_bed_risk_score=("bed_risk_score", "mean"),
      )
      .reset_index()
      .sort_values("avg_bed_risk_score", ascending=False)
)

dashboard_view["risk_tier"] = pd.cut(
    dashboard_view["avg_bed_risk_score"],
    bins=[-0.01, RISK_THRESHOLD_MED, RISK_THRESHOLD_HIGH, 1.01],
    labels=["Low", "Medium", "High"]
)

print("Top 10 highest-risk department/hour windows:")
dashboard_view.head(10)


In [ ]:
out_path = "/content/dashboard_view.csv"
dashboard_view.to_csv(out_path, index=False)
print(f"Saved dashboard feed to {out_path} -- this is what Step 3 (UI) reads from.")


## 6. Next steps (Step 2 of the build plan)

1. **Scale up**: pull a large slice of `bigquery-public-data.cms_medicare` via BigQuery (or a big synthetic multiplier of this dataset) to make the CPU-vs-GPU gap real and dramatic, not a rounding error.
2. **Flip the toggle**: set `USE_CUDF = True` in the config cell above after `pip install cudf-cu12`, restart the runtime, and re-run this same notebook unchanged.
3. **Time both runs** with `time.perf_counter()` around the feature-engineering + risk-scoring cells, and capture pandas-seconds vs. cudf.pandas-seconds.
4. **Tie the speedup to the decision**, not just the benchmark: e.g. *"On the full monthly dataset, pandas takes N minutes to refresh -- long enough that a surge is already underway before the dashboard catches it. cudf.pandas refreshes in M seconds, so the charge nurse sees the risk tier flip in near-real-time and can call in staff before the ED is already overwhelmed."* That sentence is the whole point of the NVIDIA track's grading criterion -- keep it front and center in the deck and demo video.

BigQuery template for Step 2:


---
# Step 2: Scale up with BigQuery CMS Medicare data + cuDF benchmark

**Honest note before you build the deck slide on this:** `bigquery-public-data.cms_medicare` (inpatient/outpatient charges) is **provider-level, DRG-level annual data** -- one row per hospital per diagnosis-related-group per year. It does **not** have per-patient visit timestamps, so it can't plug into the same "rolling wait time per hour" logic as Step 1's ER data.

Rather than force a fake fit, we use it for what it's actually good for here: a **real, free, large-scale dataset** (2011-2018, ~3,000 hospitals x ~500-750 DRGs x up to 8 years) to stress-test the *same pipeline architecture* -- BigQuery ingestion -> groupby/rolling feature engineering -> weighted risk score -- at a scale where the CPU-vs-GPU gap becomes dramatic and undeniable. Say this plainly in the deck: *"CMS Medicare data validates that our cuDF-accelerated architecture scales from a single hospital's daily ER feed to a nationwide, multi-year claims volume -- the same operations, the same speedup."*


In [ ]:
# Run this cell in Colab with a GCP project that has BigQuery API enabled.
# Uses the free 1 TB/month query tier -- this query is well under that.

# from google.cloud import bigquery
# client = bigquery.Client(project="YOUR_PROJECT_ID")
#
# query = '''
#     SELECT
#         provider_id,
#         provider_state,
#         drg_definition,
#         total_discharges,
#         average_covered_charges,
#         average_total_payments,
#         average_medicare_payments,
#         2014 AS year
#     FROM `bigquery-public-data.cms_medicare.inpatient_charges_2014`
#     UNION ALL
#     SELECT provider_id, provider_state, drg_definition, total_discharges,
#            average_covered_charges, average_total_payments, average_medicare_payments, 2015
#     FROM `bigquery-public-data.cms_medicare.inpatient_charges_2015`
#     UNION ALL
#     SELECT provider_id, provider_state, drg_definition, total_discharges,
#            average_covered_charges, average_total_payments, average_medicare_payments, 2016
#     FROM `bigquery-public-data.cms_medicare.inpatient_charges_2016`
# '''
# cms_df = client.query(query).to_dataframe()
# print(cms_df.shape)

# --- Fallback: synthetic data with the same real schema, so this cell runs today
# without GCP credentials. Swap this out for the real query above once you have
# BigQuery access set up (takes ~5 min: enable API, `pip install google-cloud-bigquery`,
# `gcloud auth application-default login` in Colab).

def load_cms_data(n_rows=300_000, seed=7):
    rng = np.random.default_rng(seed)
    n_providers, n_drgs = 3000, 400
    base_charge = rng.gamma(shape=3.0, scale=8000, size=n_rows)
    return pd.DataFrame({
        "provider_id": rng.integers(100000, 100000 + n_providers, size=n_rows),
        "drg_code": rng.integers(1, n_drgs, size=n_rows),
        "year": rng.choice([2014, 2015, 2016, 2017], size=n_rows),
        "total_discharges": rng.integers(11, 500, size=n_rows),
        "average_covered_charges": base_charge * rng.uniform(1.5, 3.0, size=n_rows),
        "average_total_payments": base_charge * rng.uniform(0.8, 1.2, size=n_rows),
        "average_medicare_payments": (base_charge * rng.uniform(0.8, 1.2, size=n_rows)) * rng.uniform(0.85, 0.98, size=n_rows),
    })

cms_df = load_cms_data()
print(cms_df.shape)
cms_df.head()


## Feature engineering at scale

Same *shape* of operations as Step 1 -- groupby + transform + rolling + rank -- just against the CMS schema. This is the code that gets timed under pandas vs. `cudf.pandas` below. Written as one function so both benchmark runs call the exact same logic.


In [ ]:
def run_feature_pipeline(input_df):
    d = input_df.copy()
    d = d.sort_values(["provider_id", "year"]).reset_index(drop=True)

    # national benchmark per DRG -- "what should this procedure typically cost"
    d["drg_national_avg_payment"] = d.groupby("drg_code")["average_total_payments"].transform("mean")
    d["cost_deviation_ratio"] = d["average_total_payments"] / d["drg_national_avg_payment"]

    # provider-level rolling cost trend across years -- mirrors the rolling wait-time
    # average from Step 1: "is this provider's cost trending up recently"
    d["rolling_provider_cost_avg"] = (
        d.groupby("provider_id")["average_total_payments"]
         .transform(lambda s: s.rolling(window=5, min_periods=1).mean())
    )

    # markup ratio -- covered charges vs. what Medicare actually paid
    d["markup_ratio"] = d["average_covered_charges"] / d["average_medicare_payments"]

    # discharge volume percentile within DRG -- mirrors the arrivals-per-hour load signal
    d["discharge_volume_rank"] = d.groupby("drg_code")["total_discharges"].rank(pct=True)

    def normalize(s):
        lo, hi = s.min(), s.max()
        return (s - lo) / (hi - lo) if hi > lo else s * 0

    d["risk_cost_deviation"] = normalize(d["cost_deviation_ratio"])
    d["risk_markup"] = normalize(d["markup_ratio"])
    d["risk_volume"] = normalize(d["discharge_volume_rank"])

    weights = {"risk_cost_deviation": 0.40, "risk_markup": 0.35, "risk_volume": 0.25}
    d["cost_risk_score"] = sum(d[c] * w for c, w in weights.items())
    return d

# quick correctness check before timing
_check = run_feature_pipeline(cms_df.sample(5000, random_state=1))
_check[["provider_id", "drg_code", "cost_risk_score"]].head()


## Benchmark harness

Runs the same pipeline at increasing row counts and records wall-clock time. **Run this notebook twice in Colab**, restarting the runtime between runs:

1. First run: `USE_CUDF = False` in the Step 1 config cell -> produces pandas timings.
2. Second run: `USE_CUDF = True` (after `pip install cudf-cu12`) -> produces GPU timings.

Both runs append to `/content/benchmark_results.csv`, so the comparison cell below works after either run has happened at least once for each backend.


In [ ]:
import time

BENCHMARK_SIZES = [10_000, 50_000, 100_000, 300_000, len(cms_df)]
BENCHMARK_SIZES = sorted(set(min(n, len(cms_df)) for n in BENCHMARK_SIZES))

results = []
for n in BENCHMARK_SIZES:
    sub = cms_df.sample(n=n, random_state=1).reset_index(drop=True)
    t0 = time.perf_counter()
    _ = run_feature_pipeline(sub)
    t1 = time.perf_counter()
    backend = "cudf.pandas" if USE_CUDF else "pandas"
    results.append({"n_rows": n, "seconds": round(t1 - t0, 4), "backend": backend})
    print(f"{backend:12s} | n={n:>9,} | {t1 - t0:.4f}s")

results_df = pd.DataFrame(results)

results_path = "/content/benchmark_results.csv"
try:
    existing = pd.read_csv(results_path)
    combined = pd.concat([existing, results_df], ignore_index=True)
except FileNotFoundError:
    combined = results_df
combined.to_csv(results_path, index=False)
print(f"\nSaved to {results_path}")
combined


## Speedup chart -- the acceleration proof

This is the evidence slide. Run after you have **both** a pandas run and a cudf.pandas run recorded in `benchmark_results.csv`.


In [ ]:
import matplotlib.pyplot as plt

bench = pd.read_csv("/content/benchmark_results.csv")
pivot = bench.pivot_table(index="n_rows", columns="backend", values="seconds", aggfunc="mean").sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_xlabel("Rows processed")
ax.set_ylabel("Pipeline runtime (seconds)")
ax.set_title("ER risk-pipeline runtime: pandas vs. cudf.pandas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("/content/speedup_chart.png", dpi=150)
plt.show()

if "pandas" in pivot.columns and "cudf.pandas" in pivot.columns:
    pivot["speedup_x"] = pivot["pandas"] / pivot["cudf.pandas"]
    print(pivot[["pandas", "cudf.pandas", "speedup_x"]])


## Tying it back to the decision (put this sentence in the deck)

*"At full CMS-scale (N rows), the pandas pipeline takes T_pandas seconds to refresh; cudf.pandas does the identical computation in T_gpu seconds -- an X-times speedup. For the charge nurse, that's the difference between a dashboard that's already stale by the time it loads during a real surge, and one that reflects the ED's current state in near-real-time -- catching a bed-risk spike while there's still time to call in staff or open overflow beds, instead of after the surge has already hit."*

Fill in the actual T_pandas / T_gpu / X numbers from your two runs above before it goes in the deck -- don't estimate them.
